In [ ]:
print("\n" + "=" * 70)
print("🎯 FINAL ANSWERS FOR TEST 4")
print("=" * 70)

print(f"\nQ1: Precision decreases, recall increases — ✓")
print(f"Q2: 120 PLN (revenue - cost) — ✓")
print(f"Q3: Cost of FP outweighs TP gain — ✓")
print(f"Q4: % of positives captured at top X% — ✓")
print(f"Q5: Cost of FP >> FN — ✓")
print(f"\nQ6: Expected Revenue = {EXPECTED_REVENUE:.0f} PLN")
print(f"Q7: 'Contact Everyone' = {profit_all:.0f} PLN")
print(f"Q8: LR Profit at 0.5 = {profit_lr_05:.0f} PLN")
print(f"Q9: TP gain < FP loss, but lower threshold captures more TPs — ✓")
print(f"Q10: Lift at 20% = {pct_lapsed_captured:.0f}%")

## Step 8: Summary of All Key Values

In [ ]:
print("\n" + "=" * 70)
print("Q10: RF Lift at 20% Contact Level")
print("=" * 70)

# Sort test customers by RF predicted probability (descending)
sorted_indices = np.argsort(-y_prob_rf)
y_test_sorted = np.asarray(y_test)[sorted_indices]

total_lapsed = y_test_sorted.sum()
cum_lapsed = np.cumsum(y_test_sorted)

# At 20% contact
pct_contact = 20
idx_20 = int(n_test * pct_contact / 100) - 1
lapsed_captured_20 = cum_lapsed[idx_20]
pct_lapsed_captured = (lapsed_captured_20 / total_lapsed) * 100

print(f"Total test set: {n_test} customers")
print(f"Top 20% = {int(n_test * 0.20)} customers")
print(f"Total lapsed in test: {total_lapsed}")
print(f"Lapsed in top 20%: {int(lapsed_captured_20)}")
print(f"Percentage captured: {pct_lapsed_captured:.1f}%")
print(f"\nANSWER: {pct_lapsed_captured:.0f}%")

## Step 7: Calculate Q10 - Lift at 20% Contact

In [ ]:
print("\n" + "=" * 70)
print("Q8: LR Profit at Threshold = 0.5")
print("=" * 70)

y_pred_lr_05 = (y_prob_lr >= 0.5).astype(int)
profit_lr_05, tp_lr, fp_lr, fn_lr, tn_lr = compute_profit(
    y_test, y_pred_lr_05, EXPECTED_REVENUE, CAMPAIGN_COST
)

print(f"ANSWER: {profit_lr_05:.0f} PLN")
print(f"\nBreakdown:")
print(f"  TP: {tp_lr}, FP: {fp_lr}")
print(f"  TP gain: {tp_lr} × ({EXPECTED_REVENUE:.0f} - {CAMPAIGN_COST}) = {tp_lr * (EXPECTED_REVENUE - CAMPAIGN_COST):.0f} PLN")
print(f"  FP loss: {fp_lr} × (-{CAMPAIGN_COST}) = {fp_lr * (-CAMPAIGN_COST):.0f} PLN")
print(f"  Total: {profit_lr_05:.0f} PLN")

## Step 6: Calculate Q8 - LR Profit at Threshold 0.5

In [ ]:
print("\n" + "=" * 70)
print("Q7: Total Profit from 'Contact Everyone' Strategy")
print("=" * 70)

n_test = len(y_test)
y_all_ones = np.ones(n_test, dtype=int)
profit_all, tp_all, fp_all, fn_all, tn_all = compute_profit(
    y_test, y_all_ones, EXPECTED_REVENUE, CAMPAIGN_COST
)

print(f"ANSWER: {profit_all:.0f} PLN")
print(f"\nBreakdown:")
print(f"  TP (lapsed contacted): {tp_all}")
print(f"  FP (active contacted): {fp_all}")
print(f"  TP gain: {tp_all} × ({EXPECTED_REVENUE:.0f} - {CAMPAIGN_COST}) = {tp_all * (EXPECTED_REVENUE - CAMPAIGN_COST):.0f} PLN")
print(f"  FP loss: {fp_all} × (-{CAMPAIGN_COST}) = {fp_all * (-CAMPAIGN_COST):.0f} PLN")
print(f"  Total: {profit_all:.0f} PLN")

## Step 5: Calculate Q7 - "Contact Everyone" Baseline

In [ ]:
print("=" * 70)
print("Q6: Expected Revenue (median total_spend of lapsed test)")
print("=" * 70)
print(f"ANSWER: {EXPECTED_REVENUE:.0f} PLN")
print(f"  - Median of lapsed test spend: {EXPECTED_REVENUE} PLN")

## Step 4: Calculate Q6 - Expected Revenue

In [ ]:
def compute_profit(y_true, y_pred, revenue, cost):
    """Compute total profit and confusion matrix components."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    
    total_profit = tp * (revenue - cost) + fp * (-cost)
    
    return total_profit, tp, fp, fn, tn

print("✓ Profit function defined")

## Step 3: Define Profit Function

In [ ]:
CAMPAIGN_COST = 80

# Load raw customer data to get expected revenue
cust_raw = pd.read_csv(DATA_DIR / "customers.csv")
cust_raw["total_spend_numeric"] = (
    cust_raw["total_spend"]
    .str.replace("PLN ", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Get median total_spend of lapsed test customers
lapsed_test_idx = y_test[y_test == 1].index
lapsed_test_spend = cust_raw.loc[cust_raw.index.isin(lapsed_test_idx), "total_spend_numeric"]
EXPECTED_REVENUE = lapsed_test_spend.median()

print(f"Campaign cost: {CAMPAIGN_COST} PLN")
print(f"Expected revenue (median lapsed spend): {EXPECTED_REVENUE} PLN")
print(f"TP net gain: {EXPECTED_REVENUE - CAMPAIGN_COST} PLN")
print(f"FP net loss: {CAMPAIGN_COST} PLN")

## Step 2: Business Parameters & Expected Revenue

In [ ]:
# Extract from checkpoint
X_train = checkpoint["X_train"]
X_test = checkpoint["X_test"]
y_train = checkpoint["y_train"]
y_test = checkpoint["y_test"]
y_prob_lr = checkpoint.get("y_prob_lr")
y_prob_rf = checkpoint.get("y_prob_rf")

print(f"Test set size: {len(y_test)}")
print(f"Lapse rate: {y_test.mean():.4f}")
print(f"LR predictions available: {y_prob_lr is not None}")
print(f"RF predictions available: {y_prob_rf is not None}")

In [ ]:
DATA_DIR = Path("./2. data")
CHECKPOINT_DIR = Path("./checkpoints")

# Try to find checkpoint
checkpoint_file = CHECKPOINT_DIR / "mp3_checkpoint.pkl"
if not checkpoint_file.exists():
    checkpoint_file = DATA_DIR / "checkpoints" / "checkpoint_for_mp4.pkl"

print(f"Looking for checkpoint at: {checkpoint_file}")
print(f"Exists: {checkpoint_file.exists()}")

if checkpoint_file.exists():
    with open(checkpoint_file, "rb") as f:
        checkpoint = pickle.load(f)
    print(f"✓ Checkpoint loaded! Keys: {list(checkpoint.keys())}")
else:
    print("ERROR: Checkpoint not found!")

## Step 1: Locate & Load Checkpoint

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"✓ Libraries loaded. Seed: {SEED}")

# MP4: Run Test 4 Verification

This notebook executes MP4 calculations to extract exact values for Test 4 answers.